# 01 — Configuração, Saneamento e Engenharia de Atributos (`KAN-10`, `KAN-11`, `KAN-12`)

- **Notebook da Trilha:** `notebooks/01_preparacao_dados.ipynb`
- **Tickets Jira Contemplados neste Notebook:**
  - [`KAN-10` — Criar inventário e dicionário de dados](https://megamatic.atlassian.net/browse/KAN-10)
  - [`KAN-11` — Analisar valores ausentes e duplicados](https://megamatic.atlassian.net/browse/KAN-11)
  - [`KAN-12` — Criar variáveis derivadas do Spaceship Titanic](https://megamatic.atlassian.net/browse/KAN-12)
- **Fonte dos Dados:** [Kaggle — Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic) (arquivos locais em `data/raw/`)
- **Responsável Principal (Preparação de Dados):** Gustavo
- **Revisor Dedicado (Preparação de Dados):** Helton
- **Documentação Complementar:** [`docs/dicionario_dados.md`](../docs/dicionario_dados.md) e [`docs/inventario_dados.md`](../docs/inventario_dados.md)

---

## Especificação Oficial dos Arquivos e Campos (*File and Data Field Descriptions*)

- **`train.csv`** — Registros pessoais de cerca de dois terços (~8.700) dos passageiros, utilizados como dados de treinamento e análise inferencial.
  - `PassengerId` — Identificador único de cada passageiro no formato `gggg_pp`, onde `gggg` indica o grupo com o qual o passageiro está viajando e `pp` é o seu número sequencial dentro do grupo. Pessoas em um grupo são frequentemente membros da mesma família, mas nem sempre.
  - `HomePlanet` — O planeta de onde o passageiro partiu, tipicamente seu planeta de residência permanente.
  - `CryoSleep` — Indica se o passageiro optou por ser colocado em animação suspensa durante a viagem. Passageiros em criossono ficam confinados em suas cabines.
  - `Cabin` — O número da cabine onde o passageiro está hospedado. Assume o formato `deck/num/side`, onde `side` pode ser `P` para *Port* (Bombordo) ou `S` para *Starboard* (Boreste).
  - `Destination` — O planeta para o qual o passageiro desembarcará.
  - `Age` — A idade do passageiro.
  - `VIP` — Indica se o passageiro pagou por serviço VIP especial durante a viagem.
  - `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck` — Valor faturado pelo passageiro em cada uma das diversas comodidades de luxo da *Spaceship Titanic*.
  - `Name` — O primeiro nome e o sobrenome do passageiro.
  - `Transported` — Indica se o passageiro foi teletransportado para outra dimensão. Esta é a variável alvo (*target*) que buscamos analisar e predizer.
- **`test.csv`** — Registros pessoais do um terço restante (~4.300) dos passageiros, utilizados como dados de teste para predição de `Transported`.
- **`sample_submission.csv`** — Arquivo modelo de submissão no formato correto (`PassengerId` e predição booleana `Transported`).

---

## 1. Preparação de Dados: Inventário e Dicionário de Dados (`KAN-10`)

Esta primeira etapa realiza a inspeção estrutural exaustiva das **14 variáveis originais** do conjunto de dados **Spaceship Titanic**, consolidando:
1. **Inventário físico e estrutural** dos ativos brutos (`train.csv`, `test.csv`, `sample_submission.csv`) em `data/raw/`.
2. **Classificação segundo a tipologia estatística** padronizada pela equipe:
   - **Categórica Nominal:** `HomePlanet`, `Destination`, `CryoSleep`, `VIP`, `Transported`
   - **Categórica Estruturada:** `PassengerId`, `Cabin`, `Name`
   - **Quantitativa Discreta:** `Age`
   - **Quantitativa Contínua:** `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`
3. **Tabela formal do Dicionário de Dados** contendo: *Nome da Coluna*, *Tipo do Dado*, *Tipologia Estatística*, *Escala de Medida*, *Descrição de Negócio* e *Valores Possíveis / Amplitude*.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# Fixação de semente aleatória para reprodutibilidade total
np.random.seed(42)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

# Resolução resiliente do diretório data/raw no filesystem local
RAW_DIR = Path("../data/raw") if Path("../data/raw/train.csv").exists() else Path("data/raw")
train_path = RAW_DIR / "train.csv"
test_path = RAW_DIR / "test.csv"
sub_path = RAW_DIR / "sample_submission.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_sub = pd.read_csv(sub_path)

print(f"Diretório de dados resolvido: {RAW_DIR}")
print(f"train.csv carregado com sucesso: {df_train.shape[0]:,} linhas x {df_train.shape[1]} colunas")
print(f"test.csv carregado com sucesso:  {df_test.shape[0]:,} linhas x {df_test.shape[1]} colunas")
print(f"sample_submission.csv carregado: {df_sub.shape[0]:,} linhas x {df_sub.shape[1]} colunas")


---
## 2. Inventário dos Arquivos Brutos (`data/raw/`)

Inspeção de volumetria, granularidade, integridade de chaves (`PassengerId`), duplicatas e proporção global de células ausentes nos arquivos originais da competição.


In [ ]:
inventario_arquivos = pd.DataFrame([
    {
        "Arquivo": "train.csv",
        "Caminho": "data/raw/train.csv",
        "Tamanho (Bytes)": train_path.stat().st_size,
        "Linhas (Registros)": df_train.shape[0],
        "Colunas": df_train.shape[1],
        "IDs Únicos (PassengerId)": df_train["PassengerId"].nunique(),
        "Linhas Duplicadas": int(df_train.duplicated().sum()),
        "Células Ausentes (NaN)": int(df_train.isna().sum().sum()),
        "% Células Ausentes": round(df_train.isna().sum().sum() / df_train.size * 100, 2),
        "Contém Alvo (Transported)": "Sim"
    },
    {
        "Arquivo": "test.csv",
        "Caminho": "data/raw/test.csv",
        "Tamanho (Bytes)": test_path.stat().st_size,
        "Linhas (Registros)": df_test.shape[0],
        "Colunas": df_test.shape[1],
        "IDs Únicos (PassengerId)": df_test["PassengerId"].nunique(),
        "Linhas Duplicadas": int(df_test.duplicated().sum()),
        "Células Ausentes (NaN)": int(df_test.isna().sum().sum()),
        "% Células Ausentes": round(df_test.isna().sum().sum() / df_test.size * 100, 2),
        "Contém Alvo (Transported)": "Não"
    },
    {
        "Arquivo": "sample_submission.csv",
        "Caminho": "data/raw/sample_submission.csv",
        "Tamanho (Bytes)": sub_path.stat().st_size,
        "Linhas (Registros)": df_sub.shape[0],
        "Colunas": df_sub.shape[1],
        "IDs Únicos (PassengerId)": df_sub["PassengerId"].nunique(),
        "Linhas Duplicadas": int(df_sub.duplicated().sum()),
        "Células Ausentes (NaN)": int(df_sub.isna().sum().sum()),
        "% Células Ausentes": round(df_sub.isna().sum().sum() / df_sub.size * 100, 2),
        "Contém Alvo (Transported)": "Template"
    }
])

display(inventario_arquivos)


### 2.1. Amostra Inicial dos Registros (`df_train.head()`)

In [ ]:
display(df_train.head(10))

### 2.2. Inventário Técnico de Completude e Cardinalidade por Coluna (`train.csv` e `test.csv`)

In [ ]:
inventario_colunas = []
for idx, col in enumerate(df_train.columns, start=1):
    n_train = len(df_train)
    null_tr = int(df_train[col].isna().sum())
    uniq_tr = int(df_train[col].nunique(dropna=True))
    
    if col in df_test.columns:
        null_te = int(df_test[col].isna().sum())
        pct_te = round(null_te / len(df_test) * 100, 2)
        uniq_te = int(df_test[col].nunique(dropna=True))
    else:
        null_te, pct_te, uniq_te = None, None, None

    inventario_colunas.append({
        "Ordem": idx,
        "Nome da Coluna": col,
        "Dtype (pandas)": str(df_train[col].dtype),
        "Não-Nulos (train)": n_train - null_tr,
        "Ausentes (train)": null_tr,
        "% Ausentes (train)": round(null_tr / n_train * 100, 2),
        "Valores Únicos (train)": uniq_tr,
        "Ausentes (test)": null_te,
        "% Ausentes (test)": pct_te,
        "Valores Únicos (test)": uniq_te,
    })

df_inventario_colunas = pd.DataFrame(inventario_colunas)
display(df_inventario_colunas)


---
## 3. Dicionário de Dados Formal e Tipologia Estatística (`KAN-10`)

Abaixo construímos programaticamente a tabela formal do **Dicionário de Dados** para as 14 variáveis originais do dataset **Spaceship Titanic**, contemplando:
- **Nome da Coluna**
- **Tipo do Dado**
- **Tipologia Estatística** (*Categórica Nominal*, *Categórica Estruturada*, *Quantitativa Discreta*, *Quantitativa Contínua*)
- **Escala de Medida** (*Nominal*, *Razão*)
- **Descrição de Negócio**
- **Valores Possíveis / Amplitude** (calculados diretamente sobre `train.csv`)


In [ ]:
# Extração dinâmica de amplitudes e domínios reais em train.csv
cabin_parts = df_train["Cabin"].dropna().str.split("/", expand=True)
decks_unicos = sorted(cabin_parts[0].unique().tolist())
cabin_num_min, cabin_num_max = int(cabin_parts[1].astype(int).min()), int(cabin_parts[1].astype(int).max())
sides_unicos = sorted(cabin_parts[2].unique().tolist())

pid_parts = df_train["PassengerId"].str.split("_", expand=True)
grupo_min, grupo_max = pid_parts[0].min(), pid_parts[0].max()
n_grupos = pid_parts[0].nunique()
pp_min, pp_max = pid_parts[1].min(), pid_parts[1].max()

dicionario_dados = pd.DataFrame([
    {
        "Nome da Coluna": "PassengerId",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Estruturada",
        "Escala de Medida": "Nominal (Identificador Composto)",
        "Descrição de Negócio": "Identificador único de cada passageiro no formato gggg_pp, onde gggg indica o grupo de viagem e pp é o índice sequencial do membro dentro do grupo.",
        "Valores Possíveis / Amplitude": f"Formato gggg_pp ({df_train['PassengerId'].min()} a {df_train['PassengerId'].max()}). {n_grupos} grupos distintos ({grupo_min}–{grupo_max}); índice pp de {pp_min} a {pp_max}."
    },
    {
        "Nome da Coluna": "HomePlanet",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal",
        "Descrição de Negócio": "Planeta de residência permanente / embarque de onde o passageiro partiu.",
        "Valores Possíveis / Amplitude": "Categorias: {'Earth', 'Europa', 'Mars'} (+ valores ausentes NaN)."
    },
    {
        "Nome da Coluna": "CryoSleep",
        "Tipo do Dado": "Booleano (bool / object)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal (Binária / Dicotômica)",
        "Descrição de Negócio": "Indica se o passageiro optou por permanecer em animação suspensa (criossono) durante a viagem, ficando confinado à cabine sem realizar gastos a bordo.",
        "Valores Possíveis / Amplitude": "Valores booleanos: {False, True} (+ valores ausentes NaN)."
    },
    {
        "Nome da Coluna": "Cabin",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Estruturada",
        "Escala de Medida": "Nominal (Composta: deck/num/side)",
        "Descrição de Negócio": "Código da cabine onde o passageiro está hospedado no formato deck/num/side (convés, número da cabine e lado: P = Port/Bombordo, S = Starboard/Boreste).",
        "Valores Possíveis / Amplitude": f"{df_train['Cabin'].nunique()} cabines únicas. deck em {decks_unicos}; num em [{cabin_num_min}, {cabin_num_max}]; side em {sides_unicos}."
    },
    {
        "Nome da Coluna": "Destination",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal",
        "Descrição de Negócio": "Exoplaneta de destino onde o passageiro desembarcará ao final da viagem espacial.",
        "Valores Possíveis / Amplitude": "Categorias: {'TRAPPIST-1e', '55 Cancri e', 'PSO J318.5-22'} (+ valores ausentes NaN)."
    },
    {
        "Nome da Coluna": "Age",
        "Tipo do Dado": "Numérico Inteiro (float64 com NaN)",
        "Tipologia Estatística": "Quantitativa Discreta",
        "Escala de Medida": "Razão (Discreta em anos completos)",
        "Descrição de Negócio": "Idade do passageiro em anos completos no momento da viagem (0 representa bebês menores de 1 ano).",
        "Valores Possíveis / Amplitude": f"Inteiros em [{int(df_train['Age'].min())}, {int(df_train['Age'].max())}] anos (Média: {df_train['Age'].mean():.2f}, Mediana: {df_train['Age'].median():.1f}, DP: {df_train['Age'].std():.2f})."
    },
    {
        "Nome da Coluna": "VIP",
        "Tipo do Dado": "Booleano (bool / object)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal (Binária / Dicotômica)",
        "Descrição de Negócio": "Indica se o passageiro pagou pelo serviço especial VIP de alta prioridade a bordo da espaçonave.",
        "Valores Possíveis / Amplitude": "Valores booleanos: {False, True} (+ valores ausentes NaN)."
    },
    {
        "Nome da Coluna": "RoomService",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro em pedidos de serviço de quarto (Room Service).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['RoomService'].min():.1f}, {df_train['RoomService'].max():,.1f}] (Média: {df_train['RoomService'].mean():.2f}, Mediana: {df_train['RoomService'].median():.1f})."
    },
    {
        "Nome da Coluna": "FoodCourt",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro na praça de alimentação (Food Court).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['FoodCourt'].min():.1f}, {df_train['FoodCourt'].max():,.1f}] (Média: {df_train['FoodCourt'].mean():.2f}, Mediana: {df_train['FoodCourt'].median():.1f})."
    },
    {
        "Nome da Coluna": "ShoppingMall",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro no centro comercial (Shopping Mall).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['ShoppingMall'].min():.1f}, {df_train['ShoppingMall'].max():,.1f}] (Média: {df_train['ShoppingMall'].mean():.2f}, Mediana: {df_train['ShoppingMall'].median():.1f})."
    },
    {
        "Nome da Coluna": "Spa",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro no centro de relaxamento e bem-estar (Spa).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['Spa'].min():.1f}, {df_train['Spa'].max():,.1f}] (Média: {df_train['Spa'].mean():.2f}, Mediana: {df_train['Spa'].median():.1f})."
    },
    {
        "Nome da Coluna": "VRDeck",
        "Tipo do Dado": "Numérico Real (float64)",
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Descrição de Negócio": "Valor monetário total faturado pelo passageiro no convés de simulação de realidade virtual (VRDeck).",
        "Valores Possíveis / Amplitude": f"Reais não-negativos em [{df_train['VRDeck'].min():.1f}, {df_train['VRDeck'].max():,.1f}] (Média: {df_train['VRDeck'].mean():.2f}, Mediana: {df_train['VRDeck'].median():.1f})."
    },
    {
        "Nome da Coluna": "Name",
        "Tipo do Dado": "Texto (string)",
        "Tipologia Estatística": "Categórica Estruturada",
        "Escala de Medida": "Nominal (Texto Composto: Nome + Sobrenome)",
        "Descrição de Negócio": "Nome completo do passageiro (FirstName LastName), permitindo identificar laços familiares por sobrenome.",
        "Valores Possíveis / Amplitude": f"{df_train['Name'].nunique()} nomes únicos em train.csv (2.217 sobrenomes distintos) + valores ausentes NaN."
    },
    {
        "Nome da Coluna": "Transported",
        "Tipo do Dado": "Booleano (bool)",
        "Tipologia Estatística": "Categórica Nominal",
        "Escala de Medida": "Nominal (Binária / Alvo)",
        "Descrição de Negócio": "Variável resposta (Target): indica se o passageiro foi teletransportado para outra dimensão após a colisão com a anomalia espaço-temporal.",
        "Valores Possíveis / Amplitude": "Valores booleanos sem nulos: {True (50,36%), False (49,64%)}."
    }
])

display(dicionario_dados)


---
## 4. Inspeção Detalhada por Tipologia Estatística

### 4.1. Variáveis Categóricas Nominais (`HomePlanet`, `Destination`, `CryoSleep`, `VIP`, `Transported`)
Distribuição de frequências absolutas e relativas (incluindo valores ausentes `NaN`) na base de treinamento.


In [ ]:
colunas_nominais = ["HomePlanet", "Destination", "CryoSleep", "VIP", "Transported"]

resumo_nominais = []
for col in colunas_nominais:
    vc = df_train[col].value_counts(dropna=False)
    pct = df_train[col].value_counts(dropna=False, normalize=True) * 100
    for categoria, freq in vc.items():
        resumo_nominais.append({
            "Variável": col,
            "Categoria / Valor": "NaN (Ausente)" if pd.isna(categoria) else str(categoria),
            "Frequência Absoluta (n)": int(freq),
            "Frequência Relativa (%)": round(float(pct[categoria]), 2)
        })

df_resumo_nominais = pd.DataFrame(resumo_nominais)
display(df_resumo_nominais)


### 4.2. Variáveis Categóricas Estruturadas (`PassengerId`, `Cabin`, `Name`)
Verificação empírica dos padrões estruturais que embasarão a engenharia de variáveis derivadas (`KAN-12`).


In [ ]:
# Decomposição exploratória sem alterar o dataframe original
name_parts = df_train["Name"].dropna().str.split(" ", n=1, expand=True)

resumo_estruturadas = pd.DataFrame([
    {
        "Variável": "PassengerId",
        "Padrão Estrutural": "gggg_pp (Grupo_Membro)",
        "Subcomponentes Extraíveis": "Group (gggg), MemberIndex (pp), GroupSize, IsAlone",
        "Estatísticas em train.csv": f"6.217 grupos únicos; tamanho do grupo varia de 1 a 8 passageiros."
    },
    {
        "Variável": "Cabin",
        "Padrão Estrutural": "deck/num/side",
        "Subcomponentes Extraíveis": "CabinDeck (A–G, T), CabinNum (0–1894), CabinSide (P=Bombordo, S=Boreste)",
        "Estatísticas em train.csv": f"8 decks {decks_unicos}; números de 0 a 1894; lados P ({int((cabin_parts[2]=='P').sum())}) e S ({int((cabin_parts[2]=='S').sum())})."
    },
    {
        "Variável": "Name",
        "Padrão Estrutural": "FirstName LastName",
        "Subcomponentes Extraíveis": "FirstName (Prenome), FamilyName / LastName (Sobrenome), FamilySize",
        "Estatísticas em train.csv": f"{name_parts[0].nunique()} prenomes únicos e {name_parts[1].nunique()} sobrenomes de família únicos."
    }
])

display(resumo_estruturadas)


### 4.3. Variáveis Quantitativas Discreta (`Age`) e Contínuas (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`)
Estatísticas descritivas de posição, dispersão, quantis e proporção de zeros para as variáveis numéricas.


In [ ]:
colunas_quantitativas = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

desc_quant = df_train[colunas_quantitativas].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
desc_quant["Tipologia"] = [
    "Quantitativa Discreta" if c == "Age" else "Quantitativa Contínua"
    for c in desc_quant.index
]
desc_quant["Ausentes (NaN)"] = [int(df_train[c].isna().sum()) for c in desc_quant.index]
desc_quant["Zeros (== 0)"] = [int((df_train[c] == 0).sum()) for c in desc_quant.index]
desc_quant["% Zeros (entre válidos)"] = [
    round((df_train[c] == 0).sum() / df_train[c].notna().sum() * 100, 2)
    for c in desc_quant.index
]

cols_order = [
    "Tipologia", "count", "Ausentes (NaN)", "Zeros (== 0)", "% Zeros (entre válidos)",
    "mean", "std", "min", "25%", "50%", "75%", "90%", "95%", "99%", "max"
]
display(desc_quant[cols_order].round(2))


---
## 5. Análise de Valores Ausentes, Duplicados e Estratégia de Imputação (`KAN-11`)

- **Ticket Jira:** [KAN-11 — Analisar valores ausentes e duplicados](https://megamatic.atlassian.net/browse/KAN-11)
- **Responsável Principal:** Gustavo | **Revisor Dedicado:** Helton
- **Objetivo de Negócio e Estatístico:** Garantir a integridade do banco de dados antes das análises inferenciais e da modelagem preditiva. A presença de dados faltantes sem tratamento adequado reduz o poder estatístico (caso se aplique exclusão *listwise*) e pode enviesar estimadores paramétricos e não-paramétricos caso o mecanismo gerador da ausência não seja devidamente modelado.

### 5.1. Verificação e Auditoria de Registros Duplicados (`df.duplicated().sum()`)

Avaliamos a ocorrência de duplicidades em quatro níveis de rigor:
1. **Duplicatas exatas de linhas completas** (`df.duplicated().sum()`) em `train.csv`, `test.csv` e `sample_submission.csv`.
2. **Duplicatas de chave primária** (`PassengerId.duplicated().sum()`) dentro de cada base.
3. **Sobreposição de chaves (`PassengerId`) entre `train.csv` e `test.csv`** (garantindo independência amostral entre treino e teste).
4. **Duplicatas em subconjuntos de atributos** (excluindo identificadores nominais únicos `PassengerId` e `Name`) para investigar se existem registros repetidos sob IDs distintos.


In [ ]:
# 1. Verificação de duplicatas exatas e de chaves primárias
dup_train_full = int(df_train.duplicated().sum())
dup_test_full = int(df_test.duplicated().sum())
dup_sub_full = int(df_sub.duplicated().sum())

dup_train_id = int(df_train["PassengerId"].duplicated().sum())
dup_test_id = int(df_test["PassengerId"].duplicated().sum())
intersecao_ids = len(set(df_train["PassengerId"]).intersection(set(df_test["PassengerId"])))

# 2. Verificação de linhas idênticas desconsiderando identificadores individuais (PassengerId e Name)
cols_sem_id_train = [c for c in df_train.columns if c not in ["PassengerId", "Name"]]
cols_sem_id_test = [c for c in df_test.columns if c not in ["PassengerId", "Name"]]
dup_train_sem_id = int(df_train.duplicated(subset=cols_sem_id_train).sum())
dup_test_sem_id = int(df_test.duplicated(subset=cols_sem_id_test).sum())

# Verificação de homônimos (Name duplicado entre registros válidos)
homonimos_train = int(df_train["Name"].dropna().duplicated().sum())
homonimos_test = int(df_test["Name"].dropna().duplicated().sum())

df_auditoria_duplicatas = pd.DataFrame([
    {
        "Verificação de Integridade / Duplicidade": "Linhas completas duplicadas (df.duplicated().sum())",
        "train.csv (n=8.693)": dup_train_full,
        "test.csv (n=4.277)": dup_test_full,
        "sample_submission.csv": dup_sub_full,
        "Diagnóstico Técnico": "Nenhuma linha duplicada exata em nenhuma das bases (0 ocorrências)."
    },
    {
        "Verificação de Integridade / Duplicidade": "Chave primária duplicada (PassengerId.duplicated().sum())",
        "train.csv (n=8.693)": dup_train_id,
        "test.csv (n=4.277)": dup_test_id,
        "sample_submission.csv": int(df_sub["PassengerId"].duplicated().sum()),
        "Diagnóstico Técnico": "Unicidade de 100% do identificador PassengerId."
    },
    {
        "Verificação de Integridade / Duplicidade": "Interseção de PassengerId entre train e test",
        "train.csv (n=8.693)": intersecao_ids,
        "test.csv (n=4.277)": intersecao_ids,
        "sample_submission.csv": "—",
        "Diagnóstico Técnico": "Partições de treino e teste estritamente disjuntas (sem vazamento)."
    },
    {
        "Verificação de Integridade / Duplicidade": "Perfis idênticos sem PassengerId e Name",
        "train.csv (n=8.693)": dup_train_sem_id,
        "test.csv (n=4.277)": dup_test_sem_id,
        "sample_submission.csv": "—",
        "Diagnóstico Técnico": "Coincidências legítimas de perfil (ex: familiares na mesma cabine em CryoSleep=True com gastos zero e mesma idade)."
    },
    {
        "Verificação de Integridade / Duplicidade": "Homônimos (Name repetido com PassengerId e perfil distintos)",
        "train.csv (n=8.693)": homonimos_train,
        "test.csv (n=4.277)": homonimos_test,
        "sample_submission.csv": "—",
        "Diagnóstico Técnico": "20 nomes em train (e 7 em test) ocorrem em 2 passageiros de grupos/cabines/idades distintos (homônimos naturais)."
    }
])

display(df_auditoria_duplicatas)


**Documentação do Resultado de Duplicatas:**
- A execução de `df_train.duplicated().sum()` e `df_test.duplicated().sum()` retorna **`0` registros duplicados**. Cada linha representa um indivíduo único identificado por `PassengerId`.
- As 15 ocorrências de coincidência nos demais atributos (quando omitimos `PassengerId` e `Name`) referem-se a passageiros distintos (em sua maioria companheiros da mesma cabine em `CryoSleep == True`, com gastos iguais a `0.0` e mesma idade). Portanto, **nenhum registro deve ser removido por duplicidade**.

---
### 5.2. Contagem Absoluta e Percentual de Dados Ausentes (`missing count` e `% missing`)

Abaixo calculamos a contagem absoluta (`missing count`) e o percentual (`% missing`) de valores ausentes por variável e também a distribuição de dados faltantes **por passageiro (linha)**.


In [ ]:
# Tabela consolidada de valores ausentes por variável (train.csv e test.csv)
mapa_tipologia = dict(zip(dicionario_dados["Nome da Coluna"], dicionario_dados["Tipologia Estatística"]))

tabela_ausentes = []
for col in df_train.columns:
    miss_tr = int(df_train[col].isna().sum())
    pct_tr = round(miss_tr / len(df_train) * 100, 2)
    valid_tr = int(df_train[col].notna().sum())
    
    if col in df_test.columns:
        miss_te = int(df_test[col].isna().sum())
        pct_te = round(miss_te / len(df_test) * 100, 2)
        valid_te = int(df_test[col].notna().sum())
    else:
        miss_te, pct_te, valid_te = 0, 0.0, 0
        
    tabela_ausentes.append({
        "Variável": col,
        "Tipologia Estatística": mapa_tipologia[col],
        "Válidos (train)": valid_tr,
        "missing count (train)": miss_tr,
        "% missing (train)": pct_tr,
        "Válidos (test)": valid_te if col in df_test.columns else "—",
        "missing count (test)": miss_te if col in df_test.columns else "—",
        "% missing (test)": pct_te if col in df_test.columns else "—"
    })

df_tabela_ausentes = pd.DataFrame(tabela_ausentes).sort_values(
    by="missing count (train)", ascending=False
).reset_index(drop=True)

print("=== TABELA DE VALORES AUSENTES POR VARIÁVEL (ORDENADA POR MISSING COUNT EM TRAIN) ===")
display(df_tabela_ausentes)

# Distribuição da quantidade de valores ausentes por linha (passageiro)
na_por_linha_train = df_train.isna().sum(axis=1).value_counts().sort_index()
na_por_linha_test = df_test.isna().sum(axis=1).value_counts().sort_index()

dist_linhas = []
for k in sorted(set(na_por_linha_train.index).union(set(na_por_linha_test.index))):
    n_tr = int(na_por_linha_train.get(k, 0))
    n_te = int(na_por_linha_test.get(k, 0))
    dist_linhas.append({
        "Qtd. Colunas Ausentes no Passageiro": int(k),
        "Passageiros (train)": n_tr,
        "% Passageiros (train)": round(n_tr / len(df_train) * 100, 2),
        "Passageiros (test)": n_te,
        "% Passageiros (test)": round(n_te / len(df_test) * 100, 2)
    })

df_dist_linhas_ausentes = pd.DataFrame(dist_linhas)
print("\n=== DISTRIBUIÇÃO DE DADOS AUSENTES POR PASSAGEIRO (LINHA) ===")
display(df_dist_linhas_ausentes)


**Principais Constatações sobre a Volumetria de Ausentes:**
- Das 14 variáveis de `train.csv`, **12 apresentam valores ausentes** em proporção altamente uniforme entre **`2,06%` (`Age`, 179 ausentes)** e **`2,50%` (`CryoSleep`, 217 ausentes)**. Apenas `PassengerId` e `Transported` possuem `0%` de ausência.
- Em termos globais, apenas **1,91% das células** de `train.csv` (2.324 de 121.702) estão ausentes. Contudo, como os `NaN`s estão espalhados de forma quase independente entre as 12 colunas preditoras, **24,01% dos passageiros (2.087 de 8.693)** possuem pelo menos 1 coluna ausente.
- Isso comprova por que a **exclusão de linhas incompletas (*listwise deletion* / `dropna()`) seria metodologicamente incorreta**: descartaria quase **1/4 da amostra**, reduzindo drasticamente o poder estatístico dos testes de hipóteses e inviabilizando a predição para os 996 passageiros com `NaN` em `test.csv`.

---
### 5.3. Avaliação do Mecanismo de Ausência (MCAR, MAR e MNAR)

Segundo a taxonomia clássica de Rubin (1976), os mecanismos geradores de dados ausentes dividem-se em:
1. **MCAR (*Missing Completely at Random*):** A probabilidade de um valor estar ausente é independente tanto dos valores observados quanto dos valores não observados.
2. **MAR (*Missing at Random*):** A probabilidade de ausência (ou o valor verdadeiro subjacente ao dado faltante) é condicional às variáveis observadas no banco de dados.
3. **MNAR (*Missing Not at Random*):** O mecanismo de ausência depende diretamente do próprio valor não observado mesmo após controlar pelas variáveis observadas.

A seguir realizamos três verificações empíricas fundamentais para diagnosticar o mecanismo de ausência no **Spaceship Titanic**:
1. **Verificação Estrutural `CryoSleep == True` vs. Despesas a Bordo (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`):** avaliar se passageiros em animação suspensa possuem gastos rigorosamente iguais a zero quando observados.
2. **Verificação Estrutural `Age <= 12` (Crianças) e `HomePlanet == 'Earth'` vs. Despesas e `VIP`:** avaliar restrições determinísticas de domínio.
3. **Teste Estatístico de Independência ($\chi^2$) dos Indicadores de Ausência:** verificar se a ocorrência do registro `NaN` depende do desfecho `Transported` ou de outras variáveis.


In [ ]:
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# 1. Análise de CryoSleep == True vs. Despesas a Bordo
analise_cryo = []
for status_label, subset in [
    ("CryoSleep == True (Confinados em Criossono)", df_train[df_train["CryoSleep"] == True]),
    ("CryoSleep == False (Acordados)", df_train[df_train["CryoSleep"] == False]),
    ("CryoSleep is NaN (Status Ausente)", df_train[df_train["CryoSleep"].isna()])
]:
    for col in spend_cols:
        s = subset[col]
        n_total = len(s)
        n_miss = int(s.isna().sum())
        s_valid = s.dropna()
        n_zeros = int((s_valid == 0).sum())
        n_pos = int((s_valid > 0).sum())
        analise_cryo.append({
            "Grupo CryoSleep": status_label,
            "Variável de Despesa": col,
            "Total Passageiros": n_total,
            "Ausentes (NaN)": n_miss,
            "% Ausentes": round(n_miss / n_total * 100, 2),
            "Válidos == 0.0": n_zeros,
            "% Zeros (entre válidos)": round(n_zeros / len(s_valid) * 100, 2),
            "Válidos > 0.0": n_pos,
            "Média (Válidos)": round(float(s_valid.mean()), 2),
            "Máximo (Válidos)": round(float(s_valid.max()), 1)
        })

df_analise_cryo = pd.DataFrame(analise_cryo)
print("=== 1. VERIFICAÇÃO DE CRYOSLEEP VS. DESPESAS A BORDO (TRAIN.CSV) ===")
display(df_analise_cryo)

# 2. Verificação de Regras Determinísticas Adicionais (Age <= 12, HomePlanet, Group)
kids_tr = df_train[df_train["Age"] <= 12]
earth_tr = df_train[df_train["HomePlanet"] == "Earth"]
deck_series = df_train["Cabin"].dropna().str.split("/").str[0]
df_deck_hp = pd.crosstab(deck_series, df_train.loc[deck_series.index, "HomePlanet"])

regras_dominio = pd.DataFrame([
    {
        "Regra Estrutural / Domínio": "CryoSleep == True -> Todas as 5 Despesas == 0.0",
        "Registros Observados (train)": len(df_train[df_train["CryoSleep"] == True]),
        "Violações Observadas": int((df_train.loc[df_train["CryoSleep"] == True, spend_cols].fillna(0).sum(axis=1) > 0).sum()),
        "Células NaN Recuperáveis Exatamente (train)": int(df_train.loc[df_train["CryoSleep"] == True, spend_cols].isna().sum().sum()),
        "Células NaN Recuperáveis Exatamente (test)": int(df_test.loc[df_test["CryoSleep"] == True, spend_cols].isna().sum().sum())
    },
    {
        "Regra Estrutural / Domínio": "Soma das Despesas > 0 -> CryoSleep == False",
        "Registros Observados (train)": int((df_train[spend_cols].fillna(0).sum(axis=1) > 0).sum()),
        "Violações Observadas": int(((df_train[spend_cols].fillna(0).sum(axis=1) > 0) & (df_train["CryoSleep"] == True)).sum()),
        "Células NaN Recuperáveis Exatamente (train)": int((df_train["CryoSleep"].isna() & (df_train[spend_cols].fillna(0).sum(axis=1) > 0)).sum()),
        "Células NaN Recuperáveis Exatamente (test)": int((df_test["CryoSleep"].isna() & (df_test[spend_cols].fillna(0).sum(axis=1) > 0)).sum())
    },
    {
        "Regra Estrutural / Domínio": "Age <= 12 (Crianças) -> Todas as 5 Despesas == 0.0 e VIP == False",
        "Registros Observados (train)": len(kids_tr),
        "Violações Observadas": int((kids_tr[spend_cols].fillna(0).sum(axis=1) > 0).sum() + (kids_tr["VIP"] == True).sum()),
        "Células NaN Recuperáveis Exatamente (train)": int(kids_tr[spend_cols].isna().sum().sum() + kids_tr["VIP"].isna().sum()),
        "Células NaN Recuperáveis Exatamente (test)": int(df_test.loc[df_test["Age"] <= 12, spend_cols].isna().sum().sum() + df_test.loc[df_test["Age"] <= 12, "VIP"].isna().sum())
    },
    {
        "Regra Estrutural / Domínio": "HomePlanet == 'Earth' -> VIP == False (Terráqueos nunca são VIP)",
        "Registros Observados (train)": int(earth_tr["VIP"].notna().sum()),
        "Violações Observadas": int((earth_tr["VIP"] == True).sum()),
        "Células NaN Recuperáveis Exatamente (train)": int(earth_tr["VIP"].isna().sum()),
        "Células NaN Recuperáveis Exatamente (test)": int(df_test.loc[df_test["HomePlanet"] == "Earth", "VIP"].isna().sum())
    },
    {
        "Regra Estrutural / Domínio": "Mesmo Grupo (gggg) -> Mesmo HomePlanet (Homogeneidade de Grupo)",
        "Registros Observados (train)": int(df_train["PassengerId"].str.split("_").str[0].nunique()),
        "Violações Observadas": int((df_train.groupby(df_train["PassengerId"].str.split("_").str[0])["HomePlanet"].nunique(dropna=True) > 1).sum()),
        "Células NaN Recuperáveis Exatamente (train)": "Via Grupo / Deck / Sobrenome",
        "Células NaN Recuperáveis Exatamente (test)": "Via Grupo / Deck / Sobrenome"
    }
])

print("\n=== 2. AUDITORIA DE REGRAS DETERMINÍSTICAS DE NEGÓCIO (0% DE VIOLAÇÕES NOS DADOS OBSERVADOS) ===")
display(regras_dominio)

# 3. Teste Qui-Quadrado de Independência entre Indicador de Ausência (isna) e a variável alvo Transported
testes_mcar = []
cols_com_na = [c for c in df_train.columns if df_train[c].isna().sum() > 0]
for col in cols_com_na:
    indicador_na = df_train[col].isna()
    tab_cont = pd.crosstab(indicador_na, df_train["Transported"])
    chi2, p_val, _, _ = chi2_contingency(tab_cont)
    taxa_transp_na = df_train.loc[indicador_na, "Transported"].mean() * 100
    taxa_transp_obs = df_train.loc[~indicador_na, "Transported"].mean() * 100
    testes_mcar.append({
        "Variável (Indicador isna)": col,
        "Ausentes (n)": int(indicador_na.sum()),
        "% Transported (quando Ausente)": round(taxa_transp_na, 2),
        "% Transported (quando Observado)": round(taxa_transp_obs, 2),
        "Estatística Qui-Quadrado (vs Transported)": round(chi2, 4),
        "p-valor": round(p_val, 4),
        "Decisão (alfa = 0.05)": "Independente de Transported (Compatível com MCAR)" if p_val >= 0.05 else "Diferença marginal"
    })

df_testes_mcar = pd.DataFrame(testes_mcar)
print("\n=== 3. TESTE DE INDEPENDÊNCIA ENTRE A OCORRÊNCIA DE AUSÊNCIA (ISNA) E O ALVO TRANSPORTED ===")
display(df_testes_mcar)


**Diagnóstico Metodológico sobre o Mecanismo de Ausência (MCAR, MAR, MNAR):**
1. **Ocorrência da Perda de Registro (Compatível com MCAR):**
   - A taxa de ausência é de ~2,0% a 2,5% em todas as 12 variáveis preditoras e os testes Qui-Quadrado de independência entre o indicador `isna()` de cada coluna e a variável resposta `Transported` não rejeitam a hipótese nula de independência ($p > 0,05$ na quase totalidade das variáveis). Isso indica que a falha de registro no computador de bordo da *Spaceship Titanic* ocorreu de forma essencialmente aleatória (**MCAR**).
2. **Valor Subjacente Condicional a Variáveis Observadas (Estrutura MAR / Determinística):**
   - Embora o *evento de apagamento* da célula seja aleatório, o **valor real subjacente** depende fortemente de variáveis observadas no próprio registro ou grupo familiar (**MAR estrutural**):
     - Em **`CryoSleep == True`** ($n = 3.037$ em `train.csv`), **100,0% dos valores válidos** nas 5 variáveis de despesa (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`) são **rigorosamente iguais a `0.0`** (média = `0.0`, máximo = `0.0`, zero violações). Logo, as 361 células de gastos ausentes em passageiros com `CryoSleep == True` no treino (e 169 no teste) não são incógnitas aleatórias, mas sim **`0.0` determinístico**.
     - Reciprocamente, entre os 217 passageiros com `CryoSleep` ausente em `train.csv`, **119 possuem soma de gastos observados $> 0$**, o que implica deterministicamente **`CryoSleep = False`**.
     - Em **`Age <= 12`** ($n = 806$ crianças em `train.csv`), **100,0% dos valores válidos** possuem despesas iguais a `0.0` e `VIP == False`.
     - Em **`HomePlanet == 'Earth'`** ($n = 4.602$), **100,0% dos valores válidos** possuem `VIP == False`.
     - Na estrutura de grupos (**`Group = gggg`** extraído de `PassengerId`), **100,0% dos grupos com mais de um passageiro** compartilham o mesmo `HomePlanet`, e passageiros com `CabinDeck` em `{A, B, C, T}` pertencem 100% a `Europa`, enquanto `CabinDeck == 'G'` pertence 100% a `Earth`.

---
### 5.4. Definição e Implementação da Estratégia de Imputação (`KAN-11`)

Com base nas evidências empíricas acima, implementamos uma **estratégia de imputação hierárquica em duas camadas** (ajustada sobre `train.csv` para evitar *data leakage* e aplicada de forma consistente em `train` e `test`), garantindo **zero valores ausentes (`NaN`)** em todas as colunas:

1. **Camada 1 — Regras de Negócio Explícitas e Inferência Relacional (Grupo / Deck / Sobrenome):**
   - **Regra R1 (`CryoSleep == True` $\rightarrow$ Despesas $= 0.0$):** Todas as despesas ausentes de passageiros em criossono recebem `0.0`.
   - **Regra R2 (`Age <= 12` $\rightarrow$ Despesas $= 0.0$ e `VIP = False`):** Crianças de 0 a 12 anos têm despesas faltantes preenchidas com `0.0` e `VIP` faltante preenchido com `False`.
   - **Regra R3 (Soma de Despesas $> 0$ $\rightarrow$ `CryoSleep = False`):** Passageiros com faturamento positivo em qualquer comodidade têm `CryoSleep` ausente preenchido com `False`.
   - **Regra R4 (Inferência de `HomePlanet` por Grupo, Deck da Cabine e Sobrenome):** `HomePlanet` ausente é recuperado pelo `Group` (`gggg`), pelo `CabinDeck` (`A, B, C, T` $\rightarrow$ `Europa`; `G` $ightarrow$ `Earth`) e pelo sobrenome familiar (`LastName` quando unívoco).
   - **Regra R5 (`HomePlanet == 'Earth'` $\rightarrow$ `VIP = False`):** Passageiros da Terra têm `VIP` ausente preenchido com `False`.
   - **Regra R6 (Inferência de `Cabin` por Grupo `gggg`):** Passageiros sem cabine que viajam em grupo têm `Cabin` imputada pela cabine compartilhada de seu grupo familiar.

2. **Camada 2 — Imputação Estatística Condicional (Mediana Agrupada para Quantitativas e Moda para Categóricas):**
   - **`Age` (Quantitativa Discreta):** Imputação pela **mediana de `Age` agrupada por `(HomePlanet, VIP)`** calculada em `train.csv` (arredondada para inteiro discreto). Após imputar `Age`, reaplica-se a Regra R2 caso algum novo passageiro tenha idade $\le 12$.
   - **`CryoSleep` (Categórica Nominal Binária — casos remanescentes com gasto total zero):** Para passageiros com gasto total observado igual a zero e `CryoSleep` ainda ausente, imputa-se a **moda condicional de `CryoSleep` dado `HomePlanet` entre passageiros sem gastos** (e caso receba `True`, eventuais despesas faltantes recebem `0.0`).
   - **`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck` (Quantitativas Contínuas):** Imputação pela **mediana agrupada por `(HomePlanet, CryoSleep, VIP)`** calculada em `train.csv`. Por serem distribuições fortementes assimétricas à direita e inflacionadas de zeros, a mediana preserva a tendência central típica sem introduzir gastos artificiais nas caudas.
   - **`HomePlanet`, `Destination`, `VIP` (Categóricas Nominais remanescentes):** Imputação pela **moda** de `train.csv` (`HomePlanet = 'Earth'`, `Destination = 'TRAPPIST-1e'`, `VIP = False`).
   - **`Cabin` e `Name` (Categóricas Estruturadas remanescentes):** Para garantir 100% de completude sem valores `NaN` e preservar a estrutura `deck/num/side` para o ticket `KAN-12`, cabines ainda ausentes recebem a cabine modal do respectivo `HomePlanet`, e nomes ausentes recebem `'Unknown <LastName_do_Grupo>'` (quando o grupo possui sobrenome conhecido) ou `'Unknown Unknown'`.


In [ ]:
def executar_pipeline_imputacao(df_tr_raw: pd.DataFrame, df_te_raw: pd.DataFrame):
    """
    Aplica o pipeline hierárquico de imputação (Regras de Negócio + Mediana Agrupada + Moda)
    ajustando todos os parâmetros estatísticos exclusivamente em df_tr_raw (sem data leakage).
    Retorna (df_tr_clean, df_te_clean, df_log_auditoria).
    """
    df_tr = df_tr_raw.copy()
    df_te = df_te_raw.copy()
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    log_etapas = []

    def registrar_etapa(nome_etapa, antes_tr, depois_tr, antes_te, depois_te, Regra):
        log_etapas.append({
            "Etapa / Regra de Imputação": nome_etapa,
            "Tipo de Tratamento": Regra,
            "Células Imputadas (train)": int(antes_tr - depois_tr),
            "NaNs Restantes (train)": int(depois_tr),
            "Células Imputadas (test)": int(antes_te - depois_te),
            "NaNs Restantes (test)": int(depois_te),
        })

    # Extração auxiliar de Group e LastName sem qualquer vazamento entre train e test
    tr_obs = df_tr.copy()
    tr_obs["_Group"] = tr_obs["PassengerId"].str.split("_").str[0]
    tr_obs["_LastName"] = tr_obs["Name"].str.split(" ").str[-1]

    te_obs = df_te.copy()
    te_obs["_Group"] = te_obs["PassengerId"].str.split("_").str[0]
    te_obs["_LastName"] = te_obs["Name"].str.split(" ").str[-1]

    # Mapeamento unívoco Group -> HomePlanet (grupos de train e test são 100% disjuntos)
    grp_hp_map = {
        **tr_obs.dropna(subset=["HomePlanet"]).groupby("_Group")["HomePlanet"].agg(lambda s: s.mode().iloc[0]).to_dict(),
        **te_obs.dropna(subset=["HomePlanet"]).groupby("_Group")["HomePlanet"].agg(lambda s: s.mode().iloc[0]).to_dict(),
    }
    # Mapeamento unívoco LastName -> HomePlanet ajustado exclusivamente em train.csv (sem data leakage)
    ln_hp_counts = tr_obs.dropna(subset=["_LastName", "HomePlanet"]).groupby("_LastName")["HomePlanet"].nunique()
    univoque_lns = ln_hp_counts[ln_hp_counts == 1].index
    ln_hp_map = (
        tr_obs[tr_obs["_LastName"].isin(univoque_lns)]
        .dropna(subset=["HomePlanet"])
        .groupby("_LastName")["HomePlanet"]
        .first()
        .to_dict()
    )
    # Mapeamento Group -> Cabin e Group -> LastName dentro de cada grupo
    grp_cabin_map = {
        **tr_obs.dropna(subset=["Cabin"]).groupby("_Group")["Cabin"].agg(lambda s: s.mode().iloc[0]).to_dict(),
        **te_obs.dropna(subset=["Cabin"]).groupby("_Group")["Cabin"].agg(lambda s: s.mode().iloc[0]).to_dict(),
    }
    grp_ln_map = {
        **tr_obs.dropna(subset=["_LastName"]).groupby("_Group")["_LastName"].agg(lambda s: s.mode().iloc[0]).to_dict(),
        **te_obs.dropna(subset=["_LastName"]).groupby("_Group")["_LastName"].agg(lambda s: s.mode().iloc[0]).to_dict(),
    }

    # --- REGRA 1: CryoSleep == True -> Despesas ausentes = 0.0 ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    for df in (df_tr, df_te):
        mask_cryo = df["CryoSleep"] == True
        df.loc[mask_cryo, spend_cols] = df.loc[mask_cryo, spend_cols].fillna(0.0)
    registrar_etapa(
        "1. CryoSleep == True -> Despesas = 0.0",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Regra de Negócio Determinística"
    )

    # --- REGRA 2: Age <= 12 -> Despesas ausentes = 0.0 e VIP = False ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    for df in (df_tr, df_te):
        mask_kid = df["Age"] <= 12
        df.loc[mask_kid, spend_cols] = df.loc[mask_kid, spend_cols].fillna(0.0)
        df.loc[mask_kid & df["VIP"].isna(), "VIP"] = False
    registrar_etapa(
        "2. Age <= 12 -> Despesas = 0.0 e VIP = False",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Regra de Negócio Determinística"
    )

    # --- REGRA 3: Soma de Despesas > 0 -> CryoSleep = False ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    for df in (df_tr, df_te):
        soma_gastos = df[spend_cols].fillna(0.0).sum(axis=1)
        df.loc[df["CryoSleep"].isna() & (soma_gastos > 0), "CryoSleep"] = False
    registrar_etapa(
        "3. Soma de Despesas > 0 -> CryoSleep = False",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Regra de Negócio Determinística"
    )

    # --- REGRA 4: Inferência de Cabin e HomePlanet via Grupo (gggg), Deck e Sobrenome ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    deck_hp_map = {"A": "Europa", "B": "Europa", "C": "Europa", "T": "Europa", "G": "Earth"}
    for df in (df_tr, df_te):
        grupos = df["PassengerId"].str.split("_").str[0]
        sobrenomes = df["Name"].str.split(" ").str[-1]
        # Imputar Cabin pelo Grupo quando ausente
        mask_cab_na = df["Cabin"].isna()
        df.loc[mask_cab_na, "Cabin"] = grupos[mask_cab_na].map(grp_cabin_map)
        # Imputar HomePlanet pelo Grupo
        mask_hp_na = df["HomePlanet"].isna()
        df.loc[mask_hp_na, "HomePlanet"] = grupos[mask_hp_na].map(grp_hp_map)
        # Imputar HomePlanet pelo Deck da Cabine (A,B,C,T -> Europa; G -> Earth)
        mask_hp_na = df["HomePlanet"].isna() & df["Cabin"].notna()
        decks = df.loc[mask_hp_na, "Cabin"].str.split("/").str[0]
        df.loc[mask_hp_na, "HomePlanet"] = decks.map(deck_hp_map)
        # Imputar HomePlanet pelo Sobrenome unívoco
        mask_hp_na = df["HomePlanet"].isna() & sobrenomes.notna()
        df.loc[mask_hp_na, "HomePlanet"] = sobrenomes[mask_hp_na].map(ln_hp_map)
    registrar_etapa(
        "4. Inferência Relacional (Cabin por Grupo; HomePlanet por Grupo/Deck/Sobrenome)",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Inferência Estrutural / Relacional"
    )

    # --- REGRA 5: HomePlanet e Destination remanescentes pela Moda de train.csv ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    moda_hp = df_tr["HomePlanet"].mode().iloc[0]
    moda_dest = df_tr["Destination"].mode().iloc[0]
    for df in (df_tr, df_te):
        df["HomePlanet"] = df["HomePlanet"].fillna(moda_hp)
        df["Destination"] = df["Destination"].fillna(moda_dest)
    registrar_etapa(
        f"5. Imputação de HomePlanet ('{moda_hp}') e Destination ('{moda_dest}') pela Moda",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Moda (Categóricas Nominais)"
    )

    # --- REGRA 6: HomePlanet == 'Earth' -> VIP = False + Moda para VIP remanescente ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    moda_vip = bool(df_tr["VIP"].dropna().mode().iloc[0])
    for df in (df_tr, df_te):
        df.loc[(df["HomePlanet"] == "Earth") & df["VIP"].isna(), "VIP"] = False
        df["VIP"] = df["VIP"].fillna(moda_vip).astype(bool)
    registrar_etapa(
        f"6. VIP ausente -> False em Earth + Moda Global ({moda_vip})",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Regra de Negócio + Moda"
    )

    # --- REGRA 7: Imputação de Age pela Mediana Agrupada por (HomePlanet, VIP) ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    mediana_age_grupo = df_tr.groupby(["HomePlanet", "VIP"])["Age"].median().round().to_dict()
    mediana_age_global = float(round(df_tr["Age"].median()))
    for df in (df_tr, df_te):
        mask_age_na = df["Age"].isna()
        chaves = list(zip(df.loc[mask_age_na, "HomePlanet"], df.loc[mask_age_na, "VIP"]))
        df.loc[mask_age_na, "Age"] = [mediana_age_grupo.get(k, mediana_age_global) for k in chaves]
        # Caso algum passageiro tenha Age <= 12 após imputação, garantir gastos ausentes = 0
        mask_kid = df["Age"] <= 12
        df.loc[mask_kid, spend_cols] = df.loc[mask_kid, spend_cols].fillna(0.0)
    registrar_etapa(
        "7. Imputação de Age por Mediana Agrupada (HomePlanet, VIP)",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Mediana Agrupada (Quantitativa Discreta)"
    )

    # --- REGRA 8: CryoSleep remanescente (passageiros com gasto 0) + Despesas em novos CryoSleep ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    # Entre passageiros com gasto observado == 0, qual a moda de CryoSleep por HomePlanet no treino?
    soma_tr = df_tr[spend_cols].fillna(0.0).sum(axis=1)
    moda_cryo_zero_por_hp = (
        df_tr[soma_tr == 0]
        .dropna(subset=["CryoSleep"])
        .groupby("HomePlanet")["CryoSleep"]
        .agg(lambda s: bool(s.mode().iloc[0]))
        .to_dict()
    )
    moda_cryo_global = bool(df_tr["CryoSleep"].dropna().mode().iloc[0])
    for df in (df_tr, df_te):
        mask_cryo_na = df["CryoSleep"].isna()
        df.loc[mask_cryo_na, "CryoSleep"] = (
            df.loc[mask_cryo_na, "HomePlanet"].map(moda_cryo_zero_por_hp).fillna(moda_cryo_global)
        )
        df["CryoSleep"] = df["CryoSleep"].astype(bool)
        # Se foi imputado como CryoSleep == True, qualquer despesa ainda ausente é 0.0
        df.loc[df["CryoSleep"] == True, spend_cols] = df.loc[df["CryoSleep"] == True, spend_cols].fillna(0.0)
    registrar_etapa(
        "8. Imputação de CryoSleep (Moda Condicional dado Gasto=0 e HomePlanet) + Zeros Associados",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Moda Condicional + Regra de Negócio"
    )

    # --- REGRA 9: Despesas remanescentes (CryoSleep == False) por Mediana Agrupada (HomePlanet, CryoSleep, VIP) ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    for col in spend_cols:
        med_grp = df_tr.groupby(["HomePlanet", "CryoSleep", "VIP"])[col].median().to_dict()
        med_glob = float(df_tr[col].median())
        for df in (df_tr, df_te):
            mask_na = df[col].isna()
            if mask_na.any():
                chaves = list(zip(
                    df.loc[mask_na, "HomePlanet"],
                    df.loc[mask_na, "CryoSleep"],
                    df.loc[mask_na, "VIP"]
                ))
                df.loc[mask_na, col] = [med_grp.get(k, med_glob) for k in chaves]
    registrar_etapa(
        "9. Imputação de Despesas Remanescentes por Mediana Agrupada (HomePlanet, CryoSleep, VIP)",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Mediana Agrupada (Quantitativas Contínuas)"
    )

    # --- REGRA 10: Fechamento de Categóricas Estruturadas (Cabin e Name) ---
    na_tr0, na_te0 = df_tr.isna().sum().sum(), df_te.isna().sum().sum()
    moda_cabin_por_hp = (
        df_tr.dropna(subset=["Cabin"])
        .groupby("HomePlanet")["Cabin"]
        .agg(lambda s: s.mode().iloc[0])
        .to_dict()
    )
    moda_cabin_global = df_tr["Cabin"].mode().iloc[0]
    for df in (df_tr, df_te):
        mask_cab_na = df["Cabin"].isna()
        df.loc[mask_cab_na, "Cabin"] = df.loc[mask_cab_na, "HomePlanet"].map(moda_cabin_por_hp).fillna(moda_cabin_global)

        mask_name_na = df["Name"].isna()
        grupos_na = df.loc[mask_name_na, "PassengerId"].str.split("_").str[0]
        sobrenomes_inferidos = grupos_na.map(grp_ln_map).fillna("Unknown")
        df.loc[mask_name_na, "Name"] = "Unknown " + sobrenomes_inferidos
    registrar_etapa(
        "10. Imputação Estruturada de Cabin (Moda por HomePlanet) e Name (Sobrenome do Grupo / Unknown)",
        na_tr0, df_tr.isna().sum().sum(), na_te0, df_te.isna().sum().sum(),
        "Moda Condicional / Inferência de Grupo"
    )

    return df_tr, df_te, pd.DataFrame(log_etapas)


df_train_clean, df_test_clean, df_log_imputacao = executar_pipeline_imputacao(df_train, df_test)

print("=== AUDITORIA PASSO A PASSO DO PIPELINE DE IMPUTAÇÃO (KAN-11) ===")
display(df_log_imputacao)
print(f"\nTotal de células ausentes restantes em df_train_clean: {df_train_clean.isna().sum().sum()}")
print(f"Total de células ausentes restantes em df_test_clean:  {df_test_clean.isna().sum().sum()}")


---
### 5.5. Tabela Comparativa Antes vs. Depois da Limpeza (`KAN-11`)

Para comprovar que o processo de limpeza e imputação eliminou 100% dos valores ausentes **sem distorcer as distribuições originais** das variáveis quantitativas e categóricas, apresentamos três tabelas comparativas:
1. **Comparativo Geral de Completude (Antes vs. Depois)** em todas as 14 colunas.
2. **Comparativo de Parâmetros Estatísticos das Variáveis Quantitativas** (Média, Desvio-Padrão, Mediana, IIQ e % de Zeros antes e depois da imputação).
3. **Comparativo de Proporções das Variáveis Categóricas Nominais** (Frequências absolutas e relativas antes e depois da imputação).


In [ ]:
# 1. Tabela Comparativa Geral de Completude por Coluna (Antes vs. Depois)
comp_completude = []
for col in df_train.columns:
    na_antes_tr = int(df_train[col].isna().sum())
    na_depois_tr = int(df_train_clean[col].isna().sum())
    
    if col in df_test.columns:
        na_antes_te = int(df_test[col].isna().sum())
        na_depois_te = int(df_test_clean[col].isna().sum())
        pct_antes_te = round(na_antes_te / len(df_test) * 100, 2)
        pct_depois_te = round(na_depois_te / len(df_test_clean) * 100, 2)
    else:
        na_antes_te, na_depois_te, pct_antes_te, pct_depois_te = "—", "—", "—", "—"
        
    comp_completude.append({
        "Variável": col,
        "Dtype Antes": str(df_train[col].dtype),
        "Dtype Depois": str(df_train_clean[col].dtype),
        "Ausentes Antes (train)": na_antes_tr,
        "% Ausentes Antes (train)": round(na_antes_tr / len(df_train) * 100, 2),
        "Ausentes Depois (train)": na_depois_tr,
        "% Ausentes Depois (train)": round(na_depois_tr / len(df_train_clean) * 100, 2),
        "Ausentes Antes (test)": na_antes_te,
        "Ausentes Depois (test)": na_depois_te,
    })

df_comp_completude = pd.DataFrame(comp_completude)
print("=== 1. TABELA COMPARATIVA DE COMPLETUDE E TIPAGEM (ANTES VS. DEPOIS DA LIMPEZA) ===")
display(df_comp_completude)

# 2. Tabela Comparativa de Estatísticas Descritivas das Variáveis Quantitativas (Antes vs. Depois em train.csv)
comp_quant = []
for col in colunas_quantitativas:
    s_antes = df_train[col].dropna()
    s_depois = df_train_clean[col]
    
    iqr_antes = float(s_antes.quantile(0.75) - s_antes.quantile(0.25))
    iqr_depois = float(s_depois.quantile(0.75) - s_depois.quantile(0.25))
    
    comp_quant.append({
        "Variável Quantitativa": col,
        "N Válidos Antes": len(s_antes),
        "N Válidos Depois": len(s_depois),
        "Média Antes": round(float(s_antes.mean()), 2),
        "Média Depois": round(float(s_depois.mean()), 2),
        "DP Antes": round(float(s_antes.std()), 2),
        "DP Depois": round(float(s_depois.std()), 2),
        "Mediana Antes": round(float(s_antes.median()), 2),
        "Mediana Depois": round(float(s_depois.median()), 2),
        "IIQ Antes": round(iqr_antes, 2),
        "IIQ Depois": round(iqr_depois, 2),
        "% Zeros Antes": round(float((s_antes == 0).mean() * 100), 2),
        "% Zeros Depois": round(float((s_depois == 0).mean() * 100), 2),
    })

df_comp_quant = pd.DataFrame(comp_quant)
print("\n=== 2. COMPARATIVO DE DISTRIBUIÇÃO DAS VARIÁVEIS QUANTITATIVAS (ANTES VS. DEPOIS EM TRAIN) ===")
display(df_comp_quant)

# 3. Tabela Comparativa das Variáveis Categóricas Nominais (Proporções % Válidas Antes vs. Depois em train.csv)
comp_cat = []
for col in ["HomePlanet", "Destination", "CryoSleep", "VIP", "Transported"]:
    vc_antes = df_train[col].dropna().value_counts()
    pct_antes = df_train[col].dropna().value_counts(normalize=True) * 100
    vc_depois = df_train_clean[col].value_counts()
    pct_depois = df_train_clean[col].value_counts(normalize=True) * 100
    
    for cat in vc_depois.index:
        comp_cat.append({
            "Variável Categórica": col,
            "Categoria": str(cat),
            "Contagem Antes (Válidos)": int(vc_antes.get(cat, 0)),
            "% Válidos Antes": round(float(pct_antes.get(cat, 0.0)), 2),
            "Contagem Depois (Limpo)": int(vc_depois.get(cat, 0)),
            "% Depois (Limpo)": round(float(pct_depois.get(cat, 0.0)), 2),
            "Variação Absoluta (p.p.)": round(float(pct_depois.get(cat, 0.0) - pct_antes.get(cat, 0.0)), 2)
        })

df_comp_cat = pd.DataFrame(comp_cat)
print("\n=== 3. COMPARATIVO DE PROPORÇÕES DAS VARIÁVEIS CATEGÓRICAS NOMINAIS (ANTES VS. DEPOIS EM TRAIN) ===")
display(df_comp_cat)


### 5.6. Síntese Metodológica da Limpeza e Imputação (`KAN-11`)

1. **Integridade de Duplicatas:** Confirmou-se `df.duplicated().sum() == 0` em todas as bases (`train.csv`, `test.csv` e `sample_submission.csv`) e 100% de unicidade da chave primária `PassengerId`.
2. **Resgate Determinístico por Regras de Domínio:** Mais de **35% de todas as células ausentes** (`844` de `2.324` em `train.csv` e `404` de `1.117` em `test.csv`) foram recuperadas de forma **exata e livre de erro amostral** usando restrições físicas/estruturais do domínio (`CryoSleep == True` $\rightarrow$ gastos $= 0$; `Age <= 12` $\rightarrow$ gastos $= 0$ e `VIP = False`; gastos $> 0$ $\rightarrow$ `CryoSleep = False`; homogeneidade de `HomePlanet` e `Cabin` por grupo `gggg` e `CabinDeck`).
3. **Preservação das Distribuições (Sem Distorção):**
   - A mediana de `Age` manteve-se inalterada em **`27,0` anos** (média passou de `28,83` para `28,82` anos e desvio-padrão de `14,49` para `14,35` anos).
   - As medianas das 5 variáveis de despesas permaneceram em **`0,0`**, preservando a natureza *zero-inflated* sem inflar artificialmente a variância.
   - As proporções relativas de todas as variáveis categóricas nominais (`HomePlanet`, `Destination`, `CryoSleep`, `VIP`) sofreram variação inferior a **`0,55` ponto percentual** entre a amostra original e a base limpa `df_train_clean`.
4. **Padronização de Tipos (`dtypes`):** As variáveis booleanas `CryoSleep` e `VIP`, que antes eram lidas como `object` devido à presença de `NaN`, agora possuem tipagem estrita `bool` em `df_train_clean` e `df_test_clean`, prontas para a etapa de **Engenharia de Variáveis Derivadas (`KAN-12`)**.


---
## 6. Engenharia de Variáveis Derivadas e Exportação do Dataset Tratado (`KAN-12`)

- **Ticket Jira:** [`KAN-12` — Criar variáveis derivadas do Spaceship Titanic](https://megamatic.atlassian.net/browse/KAN-12)
- **Responsável Principal:** Gustavo | **Revisor Dedicado:** Helton
- **Dependência Direta:** [`KAN-11` — Analisar valores ausentes e duplicados](https://megamatic.atlassian.net/browse/KAN-11)

### Visão de Negócio e Escopo Técnico
Com o dataset devidamente saneado e livre de valores ausentes (`df_train_clean` e `df_test_clean`), esta etapa extrai sinais estruturais e comportamentais ocultos nos identificadores compostos (`PassengerId` e `Cabin`) e consolida o perfil de consumo de bordo nas cinco comodidades pagas da *Spaceship Titanic*:

1. **Estrutura de Grupo de Viagem (`PassengerId` $\rightarrow$ `gggg_pp`):**
   - `Group` (`str`): código identificador do grupo de viagem (`gggg`, de `'0001'` a `'9280'`), preservando o formato de 4 dígitos com zeros à esquerda.
   - `GroupSize` (`int64`): quantidade total de passageiros viajando no mesmo grupo `Group` (calculado de forma independente em cada partição, sem vazamento de dados).
   - `IsAlone` (`bool`): flag binária que indica se o passageiro viaja desacompanhado (`GroupSize == 1`).
2. **Localização Espacial e Acomodação (`Cabin` $\rightarrow$ `deck/num/side`):**
   - `CabinDeck` (`str`): convés da cabine (`'A'`, `'B'`, `'C'`, `'D'`, `'E'`, `'F'`, `'G'` ou `'T'`).
   - `CabinNum` (`int64`): número sequencial da cabine dentro do convés (`0` a `1894`).
   - `CabinSide` (`str`): lado da espaçonave em que a cabine se localiza (`'P'` = *Port* / Bombordo; `'S'` = *Starboard* / Boreste).
3. **Consumo Agregado de Bordo (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`):**
   - `TotalSpend` (`float64`): despesa total acumulada pelo passageiro (`RoomService + FoodCourt + ShoppingMall + Spa + VRDeck`).
   - `HasSpent` (`bool`): flag binária que indica se o passageiro realizou algum gasto positivo a bordo (`TotalSpend > 0`).
4. **Preservação e Persistência (`data/processed/`):**
   - Todas as colunas originais são integralmente preservadas e os datasets enriquecidos são exportados para `data/processed/spaceship_titanic_clean.csv` (treino, 22 colunas) e `data/processed/spaceship_titanic_test_clean.csv` (teste, 21 colunas).

---
### 6.1. Implementação da Engenharia de Variáveis Derivadas (`KAN-12`)


In [ ]:
def criar_variaveis_derivadas(df_clean: pd.DataFrame) -> pd.DataFrame:
    """
    Constrói as 8 variáveis derivadas especificadas no ticket KAN-12 preservando
    integralmente todas as colunas originais do DataFrame tratado:
      - Group (str), GroupSize (int64), IsAlone (bool)
      - CabinDeck (str), CabinNum (int64), CabinSide (str)
      - TotalSpend (float64), HasSpent (bool)
    Operação 100% isolada na partição recebida (zero data leakage).
    """
    df = df_clean.copy()
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

    # 1. Decomposição de PassengerId (gggg_pp) -> Group, GroupSize e IsAlone
    pid_split = df["PassengerId"].astype(str).str.split("_", n=1, expand=True)
    df["Group"] = pid_split[0].astype(str)
    df["GroupSize"] = df.groupby("Group")["Group"].transform("count").astype("int64")
    df["IsAlone"] = (df["GroupSize"] == 1).astype(bool)

    # 2. Decomposição de Cabin (deck/num/side) -> CabinDeck, CabinNum e CabinSide
    cabin_split = df["Cabin"].astype(str).str.split("/", n=2, expand=True)
    df["CabinDeck"] = cabin_split[0].astype(str)
    df["CabinNum"] = cabin_split[1].astype("int64")
    df["CabinSide"] = cabin_split[2].astype(str)

    # 3. Variável sintética TotalSpend e flag binária HasSpent
    df["TotalSpend"] = df[spend_cols].sum(axis=1).astype("float64")
    df["HasSpent"] = (df["TotalSpend"] > 0.0).astype(bool)

    return df


df_train_derived = criar_variaveis_derivadas(df_train_clean)
df_test_derived = criar_variaveis_derivadas(df_test_clean)

novas_colunas = [
    "Group", "GroupSize", "IsAlone",
    "CabinDeck", "CabinNum", "CabinSide",
    "TotalSpend", "HasSpent",
]

print(f"Dimensões de df_train_derived: {df_train_derived.shape[0]:,} linhas x {df_train_derived.shape[1]} colunas (14 originais + {len(novas_colunas)} derivadas)")
print(f"Dimensões de df_test_derived:  {df_test_derived.shape[0]:,} linhas x {df_test_derived.shape[1]} colunas (13 originais + {len(novas_colunas)} derivadas)")
print("\n=== AMOSTRA DAS COLUNAS ORIGINAIS DE ORIGEM E VARIÁVEIS DERIVADAS (TRAIN) ===")
cols_exibicao = [
    "PassengerId", "Group", "GroupSize", "IsAlone",
    "Cabin", "CabinDeck", "CabinNum", "CabinSide",
    "CryoSleep", "TotalSpend", "HasSpent", "Transported",
]
display(df_train_derived[cols_exibicao].head(10))


---
### 6.2. Dicionário Técnico e Validação de Integridade das Variáveis Derivadas (Sem Vazamento de Dados)

Abaixo apresentamos:
1. O **Dicionário de Dados das 8 Variáveis Derivadas**, detalhando origem, regra matemática/estrutural, `dtype`, tipologia estatística, escala de medida e amplitude observada.
2. A **Bateria de Validação de Integridade e Ausência de *Data Leakage***, comprovando matematicamente que todas as identidades lógicas e restrições de domínio são satisfeitas em 100% dos registros de treino e teste.


In [ ]:
# 1. Dicionário de Dados Formal das 8 Variáveis Derivadas (KAN-12)
dicionario_derivadas = pd.DataFrame([
    {
        "Variável Derivada": "Group",
        "Variável de Origem": "PassengerId (gggg_pp)",
        "Regra de Construção": "PassengerId.str.split('_').str[0]",
        "Dtype": str(df_train_derived["Group"].dtype),
        "Tipologia Estatística": "Categórica Estruturada",
        "Escala de Medida": "Nominal (Código de Grupo)",
        "Ausentes (train / test)": f"{df_train_derived['Group'].isna().sum()} / {df_test_derived['Group'].isna().sum()}",
        "Valores Possíveis / Amplitude (train)": f"{df_train_derived['Group'].nunique()} grupos distintos ('{df_train_derived['Group'].min()}' a '{df_train_derived['Group'].max()}')"
    },
    {
        "Variável Derivada": "GroupSize",
        "Variável de Origem": "Group (de PassengerId)",
        "Regra de Construção": "groupby('Group')['Group'].transform('count')",
        "Dtype": str(df_train_derived["GroupSize"].dtype),
        "Tipologia Estatística": "Quantitativa Discreta",
        "Escala de Medida": "Razão (Contagem de Membros)",
        "Ausentes (train / test)": f"{df_train_derived['GroupSize'].isna().sum()} / {df_test_derived['GroupSize'].isna().sum()}",
        "Valores Possíveis / Amplitude (train)": f"Inteiros em [{df_train_derived['GroupSize'].min()}, {df_train_derived['GroupSize'].max()}] passageiros (Média: {df_train_derived['GroupSize'].mean():.2f}, Mediana: {df_train_derived['GroupSize'].median():.1f})"
    },
    {
        "Variável Derivada": "IsAlone",
        "Variável de Origem": "GroupSize",
        "Regra de Construção": "GroupSize == 1",
        "Dtype": str(df_train_derived["IsAlone"].dtype),
        "Tipologia Estatística": "Categórica Nominal (Binária)",
        "Escala de Medida": "Nominal (Dicotômica)",
        "Ausentes (train / test)": f"{df_train_derived['IsAlone'].isna().sum()} / {df_test_derived['IsAlone'].isna().sum()}",
        "Valores Possíveis / Amplitude (train)": f"True: {int(df_train_derived['IsAlone'].sum())} ({df_train_derived['IsAlone'].mean()*100:.2f}%) | False: {int((~df_train_derived['IsAlone']).sum())} ({(1-df_train_derived['IsAlone'].mean())*100:.2f}%)"
    },
    {
        "Variável Derivada": "CabinDeck",
        "Variável de Origem": "Cabin (deck/num/side)",
        "Regra de Construção": "Cabin.str.split('/').str[0]",
        "Dtype": str(df_train_derived["CabinDeck"].dtype),
        "Tipologia Estatística": "Categórica Nominal / Ordinal",
        "Escala de Medida": "Nominal / Ordinal (Convés A–G, T)",
        "Ausentes (train / test)": f"{df_train_derived['CabinDeck'].isna().sum()} / {df_test_derived['CabinDeck'].isna().sum()}",
        "Valores Possíveis / Amplitude (train)": f"8 decks: {sorted(df_train_derived['CabinDeck'].unique().tolist())}"
    },
    {
        "Variável Derivada": "CabinNum",
        "Variável de Origem": "Cabin (deck/num/side)",
        "Regra de Construção": "Cabin.str.split('/').str[1].astype(int64)",
        "Dtype": str(df_train_derived["CabinNum"].dtype),
        "Tipologia Estatística": "Quantitativa Discreta",
        "Escala de Medida": "Intervalar / Posicional (Número da Cabine)",
        "Ausentes (train / test)": f"{df_train_derived['CabinNum'].isna().sum()} / {df_test_derived['CabinNum'].isna().sum()}",
        "Valores Possíveis / Amplitude (train)": f"Inteiros em [{df_train_derived['CabinNum'].min()}, {df_train_derived['CabinNum'].max()}] (Média: {df_train_derived['CabinNum'].mean():.2f}, Mediana: {df_train_derived['CabinNum'].median():.1f})"
    },
    {
        "Variável Derivada": "CabinSide",
        "Variável de Origem": "Cabin (deck/num/side)",
        "Regra de Construção": "Cabin.str.split('/').str[2]",
        "Dtype": str(df_train_derived["CabinSide"].dtype),
        "Tipologia Estatística": "Categórica Nominal (Binária)",
        "Escala de Medida": "Nominal (P = Port/Bombordo, S = Starboard/Boreste)",
        "Ausentes (train / test)": f"{df_train_derived['CabinSide'].isna().sum()} / {df_test_derived['CabinSide'].isna().sum()}",
        "Valores Possíveis / Amplitude (train)": f"P: {int((df_train_derived['CabinSide']=='P').sum())} ({(df_train_derived['CabinSide']=='P').mean()*100:.2f}%) | S: {int((df_train_derived['CabinSide']=='S').sum())} ({(df_train_derived['CabinSide']=='S').mean()*100:.2f}%)"
    },
    {
        "Variável Derivada": "TotalSpend",
        "Variável de Origem": "RoomService, FoodCourt, ShoppingMall, Spa, VRDeck",
        "Regra de Construção": "RoomService + FoodCourt + ShoppingMall + Spa + VRDeck",
        "Dtype": str(df_train_derived["TotalSpend"].dtype),
        "Tipologia Estatística": "Quantitativa Contínua",
        "Escala de Medida": "Razão",
        "Ausentes (train / test)": f"{df_train_derived['TotalSpend'].isna().sum()} / {df_test_derived['TotalSpend'].isna().sum()}",
        "Valores Possíveis / Amplitude (train)": f"Reais em [{df_train_derived['TotalSpend'].min():.1f}, {df_train_derived['TotalSpend'].max():,.1f}] (Média: {df_train_derived['TotalSpend'].mean():.2f}, Mediana: {df_train_derived['TotalSpend'].median():.1f})"
    },
    {
        "Variável Derivada": "HasSpent",
        "Variável de Origem": "TotalSpend",
        "Regra de Construção": "TotalSpend > 0",
        "Dtype": str(df_train_derived["HasSpent"].dtype),
        "Tipologia Estatística": "Categórica Nominal (Binária)",
        "Escala de Medida": "Nominal (Dicotômica)",
        "Ausentes (train / test)": f"{df_train_derived['HasSpent'].isna().sum()} / {df_test_derived['HasSpent'].isna().sum()}",
        "Valores Possíveis / Amplitude (train)": f"True: {int(df_train_derived['HasSpent'].sum())} ({df_train_derived['HasSpent'].mean()*100:.2f}%) | False: {int((~df_train_derived['HasSpent']).sum())} ({(1-df_train_derived['HasSpent'].mean())*100:.2f}%)"
    }
])

print("=== 1. DICIONÁRIO TÉCNICO DAS 8 VARIÁVEIS DERIVADAS (KAN-12) ===")
display(dicionario_derivadas)

# 2. Auditoria Formal de Integridade e Ausência de Vazamento de Dados (Data Leakage)
intersecao_grupos = set(df_train_derived["Group"]).intersection(set(df_test_derived["Group"]))
max_pp_tr = df_train_derived["PassengerId"].str.split("_").str[1].astype(int).groupby(df_train_derived["Group"]).transform("max")
max_pp_te = df_test_derived["PassengerId"].str.split("_").str[1].astype(int).groupby(df_test_derived["Group"]).transform("max")

cabin_reconstruida_tr = df_train_derived["CabinDeck"] + "/" + df_train_derived["CabinNum"].astype(str) + "/" + df_train_derived["CabinSide"]
cabin_reconstruida_te = df_test_derived["CabinDeck"] + "/" + df_test_derived["CabinNum"].astype(str) + "/" + df_test_derived["CabinSide"]

soma_exata_tr = df_train_derived[spend_cols].sum(axis=1)
soma_exata_te = df_test_derived[spend_cols].sum(axis=1)

validacoes_kan12 = pd.DataFrame([
    {
        "Regra de Validação / Critério de Aceitação": "1. Preservação integral das colunas originais (14 no train, 13 no test)",
        "Resultado (train)": f"{all(c in df_train_derived.columns for c in df_train.columns)} ({df_train_derived.shape[1]} cols)",
        "Resultado (test)": f"{all(c in df_test_derived.columns for c in df_test.columns)} ({df_test_derived.shape[1]} cols)",
        "Status": "OK (100%)"
    },
    {
        "Regra de Validação / Critério de Aceitação": "2. Ausência total de valores nulos (NaN) nas 22/21 colunas",
        "Resultado (train)": f"{int(df_train_derived.isna().sum().sum())} NaNs",
        "Resultado (test)": f"{int(df_test_derived.isna().sum().sum())} NaNs",
        "Status": "OK (100%)"
    },
    {
        "Regra de Validação / Critério de Aceitação": "3. Disjunção de Group entre train e test (Zero Data Leakage em GroupSize)",
        "Resultado (train)": f"{df_train_derived['Group'].nunique()} grupos exclusivos",
        "Resultado (test)": f"{df_test_derived['Group'].nunique()} grupos exclusivos (Interseção = {len(intersecao_grupos)})",
        "Status": "OK (Sem Vazamento)"
    },
    {
        "Regra de Validação / Critério de Aceitação": "4. Coerência estrutural: GroupSize == maior índice sequencial (pp) do grupo",
        "Resultado (train)": f"{int((df_train_derived['GroupSize'] == max_pp_tr).sum())} / {len(df_train_derived)} (100.0%)",
        "Resultado (test)": f"{int((df_test_derived['GroupSize'] == max_pp_te).sum())} / {len(df_test_derived)} (100.0%)",
        "Status": "OK (100%)"
    },
    {
        "Regra de Validação / Critério de Aceitação": "5. Consistência lógica: IsAlone == (GroupSize == 1)",
        "Resultado (train)": f"{int((df_train_derived['IsAlone'] == (df_train_derived['GroupSize'] == 1)).sum())} / {len(df_train_derived)}",
        "Resultado (test)": f"{int((df_test_derived['IsAlone'] == (df_test_derived['GroupSize'] == 1)).sum())} / {len(df_test_derived)}",
        "Status": "OK (100%)"
    },
    {
        "Regra de Validação / Critério de Aceitação": "6. Reconstrução bijetora: CabinDeck/CabinNum/CabinSide == Cabin",
        "Resultado (train)": f"{int((cabin_reconstruida_tr == df_train_derived['Cabin']).sum())} / {len(df_train_derived)}",
        "Resultado (test)": f"{int((cabin_reconstruida_te == df_test_derived['Cabin']).sum())} / {len(df_test_derived)}",
        "Status": "OK (100%)"
    },
    {
        "Regra de Validação / Critério de Aceitação": "7. Domínio válido de CabinDeck in {A..G, T} e CabinSide in {P, S}",
        "Resultado (train)": f"Decks={sorted(df_train_derived['CabinDeck'].unique())}; Sides={sorted(df_train_derived['CabinSide'].unique())}",
        "Resultado (test)": f"Decks={sorted(df_test_derived['CabinDeck'].unique())}; Sides={sorted(df_test_derived['CabinSide'].unique())}",
        "Status": "OK (100%)"
    },
    {
        "Regra de Validação / Critério de Aceitação": "8. Exatidão algébrica de TotalSpend e equivalência HasSpent == (TotalSpend > 0)",
        "Resultado (train)": f"Erro máx = {float(np.abs(df_train_derived['TotalSpend'] - soma_exata_tr).max()):.1e}",
        "Resultado (test)": f"Erro máx = {float(np.abs(df_test_derived['TotalSpend'] - soma_exata_te).max()):.1e}",
        "Status": "OK (100%)"
    },
    {
        "Regra de Validação / Critério de Aceitação": "9. Coerência física: Passageiros em CryoSleep == True possuem HasSpent == False",
        "Resultado (train)": f"{int((df_train_derived.loc[df_train_derived['CryoSleep'], 'HasSpent']).sum())} violações",
        "Resultado (test)": f"{int((df_test_derived.loc[df_test_derived['CryoSleep'], 'HasSpent']).sum())} violações",
        "Status": "OK (0 Violações)"
    },
])

print("\n=== 2. AUDITORIA DE INTEGRIDADE E AUSÊNCIA DE VAZAMENTO DE DADOS (KAN-12) ===")
display(validacoes_kan12)


---
### 6.3. Resumo Estatístico e Sinal Explicativo das Variáveis Derivadas vs. `Transported`

Para demonstrar o valor analítico das variáveis construídas no `KAN-12` para as próximas trilhas (Estatística Descritiva, Distribuições, Testes de Hipóteses e Regressões), avaliamos a distribuição das novas variáveis e sua associação empírica com a variável alvo `Transported` na base de treinamento (`df_train_derived`):


In [ ]:
# 1. Resumo Descritivo das Variáveis Quantitativas Derivadas (GroupSize, CabinNum, TotalSpend)
quant_derivadas = ["GroupSize", "CabinNum", "TotalSpend"]
resumo_quant_deriv = []
for col in quant_derivadas:
    s = df_train_derived[col]
    resumo_quant_deriv.append({
        "Variável Derivada": col,
        "Dtype": str(s.dtype),
        "N": len(s),
        "Zeros (== 0)": int((s == 0).sum()),
        "% Zeros": round(float((s == 0).mean() * 100), 2),
        "Média": round(float(s.mean()), 2),
        "DP": round(float(s.std()), 2),
        "Mín": round(float(s.min()), 2),
        "Q1 (25%)": round(float(s.quantile(0.25)), 2),
        "Mediana (50%)": round(float(s.median()), 2),
        "Q3 (75%)": round(float(s.quantile(0.75)), 2),
        "P95": round(float(s.quantile(0.95)), 2),
        "Máx": round(float(s.max()), 2),
    })

print("=== 1. ESTATÍSTICAS DESCRITIVAS DAS VARIÁVEIS QUANTITATIVAS DERIVADAS (TRAIN) ===")
display(pd.DataFrame(resumo_quant_deriv))

# 2. Associação das Variáveis Categóricas / Binárias Derivadas com o Alvo Transported
cat_derivadas = ["IsAlone", "HasSpent", "CabinSide", "CabinDeck"]
associacao_derivadas = []
for col in cat_derivadas:
    tab = pd.crosstab(df_train_derived[col], df_train_derived["Transported"])
    chi2, p_val, dof, _ = chi2_contingency(tab)
    for categoria, row in tab.iterrows():
        n_cat = int(row.sum())
        n_transp = int(row[True])
        associacao_derivadas.append({
            "Variável Derivada": col,
            "Categoria / Valor": str(categoria),
            "Passageiros (n)": n_cat,
            "% do Total": round(n_cat / len(df_train_derived) * 100, 2),
            "Transported == True (n)": n_transp,
            "Taxa de Transporte (%)": round(n_transp / n_cat * 100, 2),
            "Qui-Quadrado Global (vs Transported)": round(float(chi2), 2),
            "p-valor": f"{p_val:.2e}",
        })

print("\n=== 2. CAPACIDADE EXPLICATIVA DAS VARIÁVEIS DERIVADAS EM RELAÇÃO A TRANSPORTED (TRAIN) ===")
display(pd.DataFrame(associacao_derivadas))


---
### 6.4. Exportação e Auditoria dos Datasets Tratados (`data/processed/`)

Conforme os critérios de aceitação do ticket `KAN-12`, exportamos o conjunto de treinamento limpo e enriquecido para `data/processed/spaceship_titanic_clean.csv` (e complementarmente o conjunto de teste tratado para `data/processed/spaceship_titanic_test_clean.csv`), validando a integridade física e estrutural dos arquivos gerados em disco.


In [ ]:
# Resolução resiliente do diretório data/processed no filesystem local
PROCESSED_DIR = Path("../data/processed") if Path("../data/raw").exists() else Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean_train_path = PROCESSED_DIR / "spaceship_titanic_clean.csv"
clean_test_path = PROCESSED_DIR / "spaceship_titanic_test_clean.csv"

df_train_derived.to_csv(clean_train_path, index=False)
df_test_derived.to_csv(clean_test_path, index=False)

# Releitura a partir do disco para auditoria de persistência
df_check_tr = pd.read_csv(clean_train_path)
df_check_te = pd.read_csv(clean_test_path)

assert df_check_tr.shape == (8693, 22), f"Shape inesperado no treino exportado: {df_check_tr.shape}"
assert df_check_te.shape == (4277, 21), f"Shape inesperado no teste exportado: {df_check_te.shape}"
assert df_check_tr.isna().sum().sum() == 0, "Foram encontrados NaNs no arquivo spaceship_titanic_clean.csv!"
assert df_check_te.isna().sum().sum() == 0, "Foram encontrados NaNs no arquivo spaceship_titanic_test_clean.csv!"

auditoria_exportacao = pd.DataFrame([
    {
        "Arquivo Exportado": "spaceship_titanic_clean.csv",
        "Caminho Relativo": "data/processed/spaceship_titanic_clean.csv",
        "Tamanho (Bytes)": clean_train_path.stat().st_size,
        "Linhas": df_check_tr.shape[0],
        "Colunas Totais": df_check_tr.shape[1],
        "Colunas Originais Preservadas": 14,
        "Variáveis Derivadas (KAN-12)": 8,
        "Células Ausentes (NaN)": int(df_check_tr.isna().sum().sum()),
        "Linhas Duplicadas": int(df_check_tr.duplicated().sum()),
    },
    {
        "Arquivo Exportado": "spaceship_titanic_test_clean.csv",
        "Caminho Relativo": "data/processed/spaceship_titanic_test_clean.csv",
        "Tamanho (Bytes)": clean_test_path.stat().st_size,
        "Linhas": df_check_te.shape[0],
        "Colunas Totais": df_check_te.shape[1],
        "Colunas Originais Preservadas": 13,
        "Variáveis Derivadas (KAN-12)": 8,
        "Células Ausentes (NaN)": int(df_check_te.isna().sum().sum()),
        "Linhas Duplicadas": int(df_check_te.duplicated().sum()),
    },
])

print(f"Arquivos exportados e validados com sucesso em: {PROCESSED_DIR}")
display(auditoria_exportacao)

print("\n=== LISTA COMPLETA DAS 22 COLUNAS DE data/processed/spaceship_titanic_clean.csv ===")
print(list(df_check_tr.columns))


### 6.5. Síntese Metodológica da Engenharia de Variáveis Derivadas (`KAN-12`)

1. **Riqueza Explicativa e Preditiva Comprovada:**
   - **`HasSpent` e `TotalSpend`:** A criação de `TotalSpend` e da flag binária `HasSpent` revelou o preditor individual de maior associação com o desfecho `Transported`: passageiros sem despesas a bordo (`HasSpent == False`, $41,92\%$ da amostra) apresentam taxa de teletransporte de **$78,73\%$**, ao passo que passageiros com despesas positivas (`HasSpent == True`, $58,08\%$ da amostra) apresentam taxa de teletransporte de apenas **$29,89\%$** ($\chi^2 = 2.017,98$, $p < 10^{-300}$).
   - **`Group` (`gggg`), `GroupSize` e `IsAlone`:** Passageiros viajando em grupo (`IsAlone == False`, $44,73\%$) têm maior probabilidade de serem teletransportados (**$56,69\%$**) do que passageiros viajando sozinhos (`IsAlone == True`, $55,27\%$, com taxa de **$45,24\%$**; $\chi^2 = 112,10$, $p = 3,39 \times 10^{-26}$).
   - **`CabinDeck`, `CabinNum` e `CabinSide`:** A decomposição de `Cabin` expôs forte estratificação espacial tanto por convés (`CabinDeck`, com taxas de transporte variando de $20,00\%$ no deck `T` e $36,05\%$ no deck `E` até $72,95\%$ no deck `B`, $\chi^2 = 396,96$, $p = 1,07 \times 10^{-81}$) quanto pelo lado da nave (`CabinSide == 'S'` / Boreste com **$55,54\%$** de transporte vs. `CabinSide == 'P'` / Bombordo com **$45,09\%$**, $\chi^2 = 94,53$, $p = 2,41 \times 10^{-22}$).
2. **Rigor contra Vazamento de Dados (*Zero Data Leakage*):**
   - Verificou-se que os $6.217$ grupos de `train.csv` e os $3.063$ grupos de `test.csv` são **100% disjuntos** (interseção nula) e completos (`GroupSize` coincide com o índice máximo `pp` em 100% dos casos), garantindo que todas as transformações de `KAN-12` operem estritamente dentro de cada partição.
3. **Critérios de Aceitação Atendidos:**
   - Os arquivos `data/processed/spaceship_titanic_clean.csv` ($8.693 \times 22$) e `data/processed/spaceship_titanic_test_clean.csv` ($4.277 \times 21$) foram gerados, validados e encontram-se prontos para alimentar as trilhas de **Estatística Descritiva (`KAN-14`/`KAN-15`)**, **Distribuições (`KAN-20`)**, **Testes de Hipóteses (`KAN-24`)** e **Correlação/Regressões (`KAN-30`/`KAN-32`)**.
